In [2]:
import pandas as pd
from pathlib import Path

# Notebook nằm trong FINAL-PROJECT/notebooks
PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DIR exists:", RAW_DIR.exists())
print("PROCESSED_DIR exists:", PROCESSED_DIR.exists())

customers = pd.read_csv(PROCESSED_DIR / "customers_clean.csv")
transactions = pd.read_csv(PROCESSED_DIR / "transactions_clean.csv")
transaction_items = pd.read_csv(PROCESSED_DIR / "transaction_items_clean.csv")
products = pd.read_csv(PROCESSED_DIR / "products_clean.csv")
stores = pd.read_csv(PROCESSED_DIR / "stores_clean.csv")
customer_support = pd.read_csv(PROCESSED_DIR / "customer_support_clean.csv")
marketing_touchpoints = pd.read_csv(PROCESSED_DIR / "marketing_touchpoints_clean.csv")

PROJECT_ROOT: /Users/blackwolf/Downloads/TTU/CS332V - Machine Learning/FInal-Project
RAW_DIR exists: True
PROCESSED_DIR exists: True


In [3]:
transactions["transaction_date"] = pd.to_datetime(transactions["transaction_date"], errors="coerce")

item_amount = (
    transaction_items.assign(item_amount=transaction_items["quantity"] * transaction_items["unit_price"])
    .groupby("transaction_id", as_index=False)["item_amount"]
    .sum()
    .rename(columns={"item_amount": "gross_amount"})
)

transactions_enriched = transactions.merge(item_amount, on="transaction_id", how="left")
transactions_enriched["gross_amount"] = transactions_enriched["gross_amount"].fillna(0)

transactions_enriched["discount_applied"] = pd.to_numeric(
    transactions_enriched["discount_applied"], errors="coerce"
).fillna(0)

transactions_enriched["total_amount"] = (
    transactions_enriched["gross_amount"] * (1 - transactions_enriched["discount_applied"])
).clip(lower=0)

print(transactions_enriched[["transaction_id", "customer_id", "transaction_date", "gross_amount", "discount_applied", "total_amount"]].head())
transactions_enriched.to_csv(PROCESSED_DIR / "transactions_enriched.csv", index=False)

  transaction_id customer_id transaction_date  gross_amount  discount_applied  \
0        T000001      C00301       2025-01-01     1710000.0              0.20   
1        T000002      C00109       2026-01-07   149082000.0              0.00   
2        T000003      C00008       2024-11-29     4836000.0              0.00   
3        T000004      C00754       2025-02-26     5687000.0              0.15   
4        T000005      C00268       2025-06-29    71016000.0              0.05   

   total_amount  
0     1368000.0  
1   149082000.0  
2     4836000.0  
3     4833950.0  
4    67465200.0  


In [4]:
snapshot_date = transactions_enriched["transaction_date"].max() + pd.Timedelta(days=1)
print("Snapshot date:", snapshot_date)

Snapshot date: 2026-03-26 00:00:00


In [5]:
rfm = (
    transactions_enriched
    .groupby("customer_id")
    .agg(
        last_purchase_date=("transaction_date", "max"),
        first_purchase_date=("transaction_date", "min"),
        frequency=("transaction_id", "nunique"),
        monetary=("total_amount", "sum"),
        avg_order_value=("total_amount", "mean")
    )
    .reset_index()
)

rfm["recency_days"] = (snapshot_date - rfm["last_purchase_date"]).dt.days
rfm["customer_lifetime_days"] = (rfm["last_purchase_date"] - rfm["first_purchase_date"]).dt.days

rfm.head()

,customer_id,last_purchase_date,first_purchase_date,frequency,monetary,avg_order_value,recency_days,customer_lifetime_days
0,C00001,2026-02-01,2024-06-04,10,81962050.0,8.196205e+06,53,607
1,C00002,2026-03-10,2025-04-17,12,230204400.0,1.918370e+07,16,327
2,C00003,2026-03-01,2023-09-09,13,52317400.0,4.024415e+06,25,904
3,C00004,2026-03-21,2025-05-10,6,131854050.0,2.197568e+07,5,315
4,C00005,2026-02-13,2024-04-23,10,99898600.0,9.989860e+06,41,661


In [7]:
# CELL 5: Features hành vi mua (category, brand, giá TB, discount_usage_rate)

# 1. Join transaction_items với products để có category/brand cho từng item
items_with_product = transaction_items.merge(
    products[["product_id", "category", "brand"]],
    on="product_id",
    how="left"
)

# 2. Thêm customer_id từ transactions_enriched
items_full = items_with_product.merge(
    transactions_enriched[["transaction_id", "customer_id"]],
    on="transaction_id",
    how="left"
)

print("Columns in items_full:", items_full.columns.tolist()[:15])

# 3. Số category & brand đã mua cho mỗi customer
category_brand_stats = (
    items_full
    .groupby("customer_id")
    .agg(
        n_unique_categories=("category", "nunique"),
        n_unique_brands=("brand", "nunique")
    )
    .reset_index()
)

# 4. Brand yêu thích (brand có tổng quantity cao nhất)
favorite_brand = (
    items_full
    .groupby(["customer_id", "brand"])["quantity"]
    .sum()
    .reset_index()
    .sort_values(["customer_id", "quantity"], ascending=[True, False])
    .drop_duplicates("customer_id")
    .rename(columns={"brand": "favorite_brand", "quantity": "favorite_brand_qty"})
)

# 5. Giá trung bình mỗi item & line amount
price_stats = (
    items_full
    .assign(line_amount=lambda df: df["quantity"] * df["unit_price"])
    .groupby("customer_id")
    .agg(
        avg_item_price=("unit_price", "mean"),
        avg_line_amount=("line_amount", "mean")
    )
    .reset_index()
)

# 6. Discount usage rate & số giao dịch từ transactions_enriched
discount_and_channel = (
    transactions_enriched
    .groupby("customer_id")
    .agg(
        discount_usage_rate=("discount_applied", lambda x: (x > 0).mean()),
        n_transactions=("transaction_id", "nunique")
    )
    .reset_index()
)

category_brand_stats.head(), favorite_brand.head(), price_stats.head(), discount_and_channel.head()

Columns in items_full: ['transaction_id', 'product_id', 'quantity', 'unit_price', 'category', 'brand', 'customer_id']


(  customer_id  n_unique_categories  n_unique_brands
 0      C00001                    6               12
 1      C00002                    6               14
 2      C00003                    6               10
 3      C00004                    6               11
 4      C00005                    5               12,
    customer_id favorite_brand  favorite_brand_qty
 3       C00001        Generic                 7.0
 18      C00002           Nike                18.0
 29      C00003    Local Brand                17.0
 42      C00004    Local Brand                 6.0
 47      C00005         Adidas                11.0,
   customer_id  avg_item_price  avg_line_amount
 0      C00001    3.274944e+06     5.100471e+06
 1      C00002    3.035407e+06     9.632280e+06
 2      C00003    9.817619e+05     2.898050e+06
 3      C00004    2.920000e+06     1.031893e+07
 4      C00005    2.122048e+06     4.952381e+06,
   customer_id  discount_usage_rate  n_transactions
 0      C00001             0.5000

In [8]:
customer_features = (
    customers[["customer_id", "gender", "birth_year", "city", "acquisition_channel"]]
    .merge(rfm, on="customer_id", how="left")
    .merge(category_brand_stats, on="customer_id", how="left")
    .merge(favorite_brand[["customer_id", "favorite_brand"]], on="customer_id", how="left")
    .merge(price_stats, on="customer_id", how="left")
    .merge(discount_and_channel, on="customer_id", how="left")
)

print(customer_features.head())
print(customer_features.shape)
customer_features.to_csv(PROCESSED_DIR / "customer_features_base.csv", index=False)
print("Saved:", PROCESSED_DIR / "customer_features_base.csv")

  customer_id gender  birth_year       city acquisition_channel  \
0      C00001      F      1979.0  Hải Phòng            Facebook   
1      C00002      M      1990.0   Biên Hòa            Referral   
2      C00003      F      1987.0        NaN              Shopee   
3      C00004      F      1966.0     TP.HCM            Referral   
4      C00005      F      1967.0     TP.HCM              Shopee   

  last_purchase_date first_purchase_date  frequency     monetary  \
0         2026-02-01          2024-06-04       10.0   81962050.0   
1         2026-03-10          2025-04-17       12.0  230204400.0   
2         2026-03-01          2023-09-09       13.0   52317400.0   
3         2026-03-21          2025-05-10        6.0  131854050.0   
4         2026-02-13          2024-04-23       10.0   99898600.0   

   avg_order_value  recency_days  customer_lifetime_days  n_unique_categories  \
0     8.196205e+06          53.0                   607.0                  6.0   
1     1.918370e+07        

In [9]:
# CELL: Features từ customer_support

customer_support["created_date"] = pd.to_datetime(customer_support["created_date"], errors="coerce")

support_agg = (
    customer_support
    .groupby("customer_id")
    .agg(
        support_ticket_count=("ticket_id", "nunique"),
        resolved_ticket_count=("resolution_status", lambda x: (x == "Resolved").sum()),
        pending_ticket_count=("resolution_status", lambda x: (x == "Pending").sum()),
        avg_satisfaction=("satisfaction_score", "mean"),
        last_ticket_date=("created_date", "max")
    )
    .reset_index()
)

snapshot_date = transactions_enriched["transaction_date"].max() + pd.Timedelta(days=1)
support_agg["days_since_last_ticket"] = (snapshot_date - support_agg["last_ticket_date"]).dt.days
support_agg["resolved_rate"] = support_agg["resolved_ticket_count"] / support_agg["support_ticket_count"]

support_agg = support_agg.drop(columns=["last_ticket_date"])
support_agg.head()

,customer_id,support_ticket_count,resolved_ticket_count,pending_ticket_count,avg_satisfaction,days_since_last_ticket,resolved_rate
0,C00003,1,1,0,5.000000,906,1.000000
1,C00004,3,2,1,5.000000,287,0.666667
2,C00005,3,3,0,3.333333,661,1.000000
3,C00006,4,3,1,4.666667,605,0.750000
4,C00009,1,0,0,4.000000,277,0.000000


In [12]:
# CELL: Features từ marketing_touchpoints

# 1. Parse cột ngày
marketing_touchpoints["sent_date"] = pd.to_datetime(
    marketing_touchpoints["sent_date"], errors="coerce"
)

# 2. Chuẩn hóa is_opened, is_clicked về 0/1
#    - chuyển sang numeric (Yes/No, TRUE/FALSE, '1'/'0' -> 1/0)
#    - NaN coi như 0 (không mở / không click)
for col in ["is_opened", "is_clicked"]:
    marketing_touchpoints[col] = pd.to_numeric(
        marketing_touchpoints[col], errors="coerce"
    ).fillna(0).astype(int)

# 3. Aggregate theo customer
marketing_agg = (
    marketing_touchpoints
    .groupby("customer_id")
    .agg(
        campaign_count=("campaign_id", "nunique"),
        total_emails=("campaign_id", "size"),
        total_open=("is_opened", "sum"),
        total_click=("is_clicked", "sum"),
        total_offer_value=("offer_value", "sum"),
        last_campaign_date=("sent_date", "max"),
    )
    .reset_index()
)

# 4. Tính các tỷ lệ
marketing_agg["open_rate"] = marketing_agg["total_open"] / marketing_agg["total_emails"]
marketing_agg["click_rate"] = marketing_agg["total_click"] / marketing_agg["total_emails"]
marketing_agg["ctr_given_open"] = (
    marketing_agg["total_click"] /
    marketing_agg["total_open"].replace(0, pd.NA)
)

# 5. Days since last campaign
marketing_agg["days_since_last_campaign"] = (
    snapshot_date - marketing_agg["last_campaign_date"]
).dt.days

marketing_agg = marketing_agg.drop(columns=["last_campaign_date"])

marketing_agg.head()

,customer_id,campaign_count,total_emails,total_open,total_click,total_offer_value,open_rate,click_rate,ctr_given_open,days_since_last_campaign
0,C00001,2,2,2,0,40000.0,1.000000,0.000000,0.0,348.0
1,C00002,3,3,1,0,70000.0,0.333333,0.000000,0.0,103.0
2,C00003,5,6,2,1,125000.0,0.333333,0.166667,0.5,38.0
3,C00004,5,5,2,1,105000.0,0.400000,0.200000,0.5,11.0
4,C00005,2,2,0,0,5000.0,0.000000,0.000000,<NA>,175.0


In [13]:
# CELL: Features thời gian (interpurchase, day-of-week, month)

tx_sorted = (
    transactions_enriched
    .sort_values(["customer_id", "transaction_date"])
    .copy()
)

tx_sorted["prev_date"] = tx_sorted.groupby("customer_id")["transaction_date"].shift(1)
tx_sorted["interpurchase_days"] = (tx_sorted["transaction_date"] - tx_sorted["prev_date"]).dt.days

time_agg = (
    tx_sorted.groupby("customer_id")
    .agg(
        mean_interpurchase_days=("interpurchase_days", "mean"),
        std_interpurchase_days=("interpurchase_days", "std"),
        last_interpurchase_gap=("interpurchase_days", "last"),
        favorite_dow=("transaction_date", lambda x: x.dt.dayofweek.mode().iloc[0] if not x.empty else pd.NA),
        favorite_month=("transaction_date", lambda x: x.dt.month.mode().iloc[0] if not x.empty else pd.NA),
    )
    .reset_index()
)

time_agg.head()

,customer_id,mean_interpurchase_days,std_interpurchase_days,last_interpurchase_gap,favorite_dow,favorite_month
0,C00001,67.444444,57.210382,11.0,0,6
1,C00002,29.727273,26.310040,76.0,0,11
2,C00003,75.333333,75.880688,32.0,6,7
3,C00004,63.000000,53.089547,124.0,5,6
4,C00005,73.444444,58.369751,64.0,1,2


In [14]:
# CELL: Merge tất cả feature levels

customer_features_full = (
    customer_features
    .merge(support_agg, on="customer_id", how="left")
    .merge(marketing_agg, on="customer_id", how="left")
    .merge(time_agg, on="customer_id", how="left")
)

print(customer_features_full.head())
print(customer_features_full.shape)

customer_features_full.to_csv(PROCESSED_DIR / "customer_features_full.csv", index=False)
print("Saved:", PROCESSED_DIR / "customer_features_full.csv")

  customer_id gender  birth_year       city acquisition_channel  \
0      C00001      F      1979.0  Hải Phòng            Facebook   
1      C00002      M      1990.0   Biên Hòa            Referral   
2      C00003      F      1987.0        NaN              Shopee   
3      C00004      F      1966.0     TP.HCM            Referral   
4      C00005      F      1967.0     TP.HCM              Shopee   

  last_purchase_date first_purchase_date  frequency     monetary  \
0         2026-02-01          2024-06-04       10.0   81962050.0   
1         2026-03-10          2025-04-17       12.0  230204400.0   
2         2026-03-01          2023-09-09       13.0   52317400.0   
3         2026-03-21          2025-05-10        6.0  131854050.0   
4         2026-02-13          2024-04-23       10.0   99898600.0   

   avg_order_value  ...  total_offer_value  open_rate  click_rate  \
0     8.196205e+06  ...            40000.0   1.000000    0.000000   
1     1.918370e+07  ...            70000.0   0.333

In [16]:
# CELL A (fixed): Tạo target label will_return_within_30_days

# Đảm bảo transaction_date là datetime
transactions_enriched["transaction_date"] = pd.to_datetime(
    transactions_enriched["transaction_date"], errors="coerce"
)

# Sắp xếp giao dịch theo customer_id, transaction_date
tx_sorted_label = transactions_enriched.sort_values(
    ["customer_id", "transaction_date"]
).copy()

# Lần mua gần nhất cho mỗi khách
last_purchase = (
    tx_sorted_label.groupby("customer_id")["transaction_date"]
    .max()
    .rename("last_purchase_date")
    .reset_index()
)

# Tạo next_transaction_date cho từng hàng
tx_sorted_label["next_transaction_date"] = tx_sorted_label.groupby("customer_id")["transaction_date"].shift(-1)

# Join last_purchase để biết với mỗi customer, last_purchase_date là gì
# Sau đó chỉ giữ hàng có transaction_date == last_purchase_date
last_with_next = (
    tx_sorted_label
    .merge(last_purchase, on="customer_id", how="left")
    .query("transaction_date == last_purchase_date")
    [["customer_id", "last_purchase_date", "next_transaction_date"]]
)

# days until next purchase
last_with_next["days_until_next_purchase"] = (
    last_with_next["next_transaction_date"] - last_with_next["last_purchase_date"]
).dt.days

# Label: quay lại trong 30 ngày
N_DAYS = 30
last_with_next["will_return_within_30_days"] = (
    last_with_next["days_until_next_purchase"].notna()
    & (last_with_next["days_until_next_purchase"] <= N_DAYS)
    & (last_with_next["days_until_next_purchase"] >= 0)
).astype(int)

print(last_with_next.head())
print(last_with_next["will_return_within_30_days"].value_counts(normalize=True))

   customer_id last_purchase_date next_transaction_date  \
9       C00001         2026-02-01                   NaT   
21      C00002         2026-03-10                   NaT   
34      C00003         2026-03-01                   NaT   
40      C00004         2026-03-21                   NaT   
50      C00005         2026-02-13                   NaT   

    days_until_next_purchase  will_return_within_30_days  
9                        NaN                           0  
21                       NaN                           0  
34                       NaN                           0  
40                       NaN                           0  
50                       NaN                           0  
will_return_within_30_days
0    0.994702
1    0.005298
Name: proportion, dtype: float64


In [17]:
# CELL B: Merge label vào customer_features_full

customer_dataset = customer_features_full.merge(
    last_with_next[["customer_id", "will_return_within_30_days", "days_until_next_purchase"]],
    on="customer_id",
    how="left"
)

# Nếu có khách không match (ít hoặc không giao dịch), gán label 0
customer_dataset["will_return_within_30_days"] = (
    customer_dataset["will_return_within_30_days"]
    .fillna(0)
    .astype(int)
)

print(customer_dataset[["customer_id", "will_return_within_30_days", "days_until_next_purchase"]].head())
print(customer_dataset["will_return_within_30_days"].value_counts())

customer_dataset.to_csv(PROCESSED_DIR / "customer_model_dataset.csv", index=False)
print("Saved:", PROCESSED_DIR / "customer_model_dataset.csv")

  customer_id  will_return_within_30_days  days_until_next_purchase
0      C00001                           0                       NaN
1      C00002                           0                       NaN
2      C00003                           0                       NaN
3      C00004                           0                       NaN
4      C00005                           0                       NaN
will_return_within_30_days
0    800
1      4
Name: count, dtype: int64
Saved: /Users/blackwolf/Downloads/TTU/CS332V - Machine Learning/FInal-Project/data/processed/customer_model_dataset.csv


In [20]:
# CELL C (final): Train/test split theo thời gian dùng last_purchase_date_x

# Kiểm tra cột thời gian hiện có
[col for col in customer_dataset.columns if "last_purchase_date" in col]
# -> sẽ thấy ['last_purchase_date_x', 'last_purchase_date_y']

# Dùng last_purchase_date_x làm mốc thời gian
customer_dataset["last_purchase_date_x"] = pd.to_datetime(
    customer_dataset["last_purchase_date_x"], errors="coerce"
)

quantile_cutoff = 0.8
time_cutoff = customer_dataset["last_purchase_date_x"].quantile(quantile_cutoff)

train_mask = customer_dataset["last_purchase_date_x"] <= time_cutoff
test_mask = customer_dataset["last_purchase_date_x"] > time_cutoff

train_df = customer_dataset[train_mask].copy()
test_df = customer_dataset[test_mask].copy()

print("Time cutoff:", time_cutoff)
print("Train size:", train_df.shape, "Test size:", test_df.shape)
print("Train positive rate:", train_df["will_return_within_30_days"].mean())
print("Test positive rate:", test_df["will_return_within_30_days"].mean())

Time cutoff: 2026-02-25 04:48:00
Train size: (604, 42) Test size: (151, 42)
Train positive rate: 0.004966887417218543
Test positive rate: 0.006622516556291391
